# Loading a Text Generation Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct",
                                             device_map = "cuda",
                                             torch_dtype = "auto",
                                             trust_remote_code = True)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct",
                                          trust_remote_code = True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [ ]:
# create a pipeline
pipe = pipeline("text-generation",
                model = model,
                tokenizer = tokenizer,
                return_full_text = False,
                do_sample = False) # sampling false means - no creativity - same ouput all time, no use of temperature and top_p

Device set to use cuda
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
# prompt
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

output = pipe(messages, use_cache=False)
print(output[0]["generated_text"])

 Why did the chicken join the band? Because it had the drumsticks!


In [ ]:
# to check the underlying tokenizer process to convert our messages into a specific promt template.
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)#, add_generation_prompt=False)
print(prompt)
# this prompt is passed directly to the llm and processed all at once.

<|user|>
Create a funny joke about chickens.<|end|>
<|endoftext|>


# Controlling model output

In [ ]:
# can use temperature in our outputs
output = pipe(messages, use_cache=False, do_sample = True, temperature=1)# model now randomely selects tokens
print(output[0]["generated_text"])

 Why don't chickens use computers? They're afraid of the egg-ceptional amount of data!


In [ ]:
# using a high top_p
output = pipe(messages, do_sample = True,use_cache=False, top_p = 5)
print(output[0]["generated_text"])

 Why did the chicken join a band? Because it wanted to be a Cluckin' Egg!


In [ ]:
#prompt components
persona = "You are and expert in Large Langugae models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary shoudl extract the most crucial points that can help researchers quickly understand the most vital informationof the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"
text = "Summarize the paper 'Attention is all you need'."
data = f"Text to summarize: {text}"

# full prompt - remove and add pieces to view its impact on the generation
query = persona + instruction + context + data_format + audience + tone + data
print(query)

You are and expert in Large Langugae models. You excel at breaking down complex papers into digestible summaries.
Summarize the key findings of the paper provided.
Your summary shoudl extract the most crucial points that can help researchers quickly understand the most vital informationof the paper.
Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.
The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.
The tone should be professional and clear.
Text to summarize: Summarize the paper 'Attention is all you need'.


In [ ]:
messages = [
    {"role": "user", "content": f"{query}"}
]

output = pipe(messages, use_cache=False)
print(output[0]["generated_text"])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


 - The paper introduces the Transformer model, which relies solely on attention mechanisms to process sequences of data.

- It demonstrates that the Transformer outperforms previous sequence-to-sequence models in various natural language processing tasks.

- The model's architecture consists of an encoder and decoder, both of which use self-attention to weigh the importance of different parts of the input data.

- The paper shows that the Transformer can be trained end-to-end for tasks like machine translation, text summarization, and question-answering.

- The model's success is attributed to its ability to handle long-range dependencies and parallelize training, making it more efficient than previous models.


The paper 'Attention is all you need' presents a groundbreaking approach to natural language processing with the introduction of the Transformer model. This model, which relies solely on attention mechanisms, outperforms previous sequence-to-sequence models in various tasks, in

# In-Context Learning :Providing examples

In [ ]:
# use a single example of using the made-up word in a sequence
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the world gugamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gugamuru that my uncle gave me as a gift. I love to play it at home."
        },
    {
        "role": "user",
        "content": "To 'Screeg' something is to swing a sword at at. An example of a sentence that uses the word screeg is:"
    }
]
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

<|user|>
A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the world gugamuru is:<|end|>
<|assistant|>
I have a Gugamuru that my uncle gave me as a gift. I love to play it at home.<|end|>
<|user|>
To 'Screeg' something is to swing a sword at at. An example of a sentence that uses the word screeg is:<|end|>
<|endoftext|>


In [ ]:
outputs = pipe(one_shot_prompt, use_cache=False)
print(outputs[0]["generated_text"])

 During the medieval reenactment, the knights would screeg at each other in a mock battle, showcasing their skills and bravery.


# Chain Prompting: breaking up the Problem

In [ ]:
# now , we create a continuous chain of intereactions that solves our problem.
# we get a sequential type of pipelineof the entire task

# create name and slogan for a prompt
product_prompt = [
    {
        "role": "user",
        "content": "Create a name and slogan for a chatbot that leverages LLMs."
    }
]
outputs = pipe(product_prompt, use_cache=False)
product_description = outputs[0]["generated_text"]
print(product_description)

 Name: ChatSage
Slogan: "Your AI Companion for Smart Conversations"


In [ ]:
# based on a name and slogan for a product, generate a sales pitch
sales_prompt = [
    {
        "role": "user",
        "content": f"Create a sales pitch for a product. Use the following product description: {product_description}"
    }
]
outputs = pipe(sales_prompt, use_cache=False)
sales_pitch = outputs[0]["generated_text"]
print(sales_pitch)

 Introducing ChatSage, your AI companion for smart conversations. With ChatSage, you can have meaningful and engaging conversations with an intelligent and friendly AI assistant. Whether you're looking for information, advice, or just someone to talk to, ChatSage is here to help.

ChatSage is designed to understand your needs and provide you with personalized responses. It can help you with a wide range of tasks, from setting reminders and managing your schedule to providing you with news updates and weather forecasts. With ChatSage, you can stay connected and informed, no matter where you are.

But ChatSage is more than just a helpful assistant. It's also a great conversation partner. With its advanced natural language processing capabilities, ChatSage can engage in intelligent and witty conversations with you. It can tell jokes, share interesting facts, and even offer advice on a variety of topics.

ChatSage is also designed to be a safe and secure platform. It uses advanced encrypti

# Reasoning with Generative Models

## chain-of-thought: Think before Answering

In [ ]:
# answering the chain of thought
cot_prompt = [
    {
        "role": "user",
        "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"
    },
    {
        "role": "assistant",
        "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. Therefore, he now has 11 balls."
    },
    {
        "role": "user",
        "content": "The cafetaria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"
    }
]

outputs = pipe(cot_prompt, use_cache=False)
print(outputs[0]["generated_text"])

 The cafetaria started with 23 apples. They used 20 apples to make lunch, so they had 23 - 20 = 3 apples left. Then they bought 6 more apples, so they now have 3 + 6 = 9 apples.


In [ ]:
# zero - shot chain of thought
zeroshot_cot_prompt = [
    {
        "role": "user",
        "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."
    }
]

outputs = pipe(zeroshot_cot_prompt, use_cache=False)
print(outputs[0]["generated_text"])

 Step 1: Start with the initial number of apples in the cafeteria, which is 23.

Step 2: Subtract the number of apples used to make lunch, which is 20.
23 - 20 = 3 apples remaining.

Step 3: Add the number of apples bought, which is 6.
3 + 6 = 9 apples.

So, the cafeteria now has 9 apples.


# Self-Consistency : Sampling Outputs

In [ ]:
# self consistency was introduced to counteract the randomeness and improve the performamnce of the generative models.
# ask the same prompt multiple times and takes the majority result as the final answer.
# does require asking the same questions multiple times.

## Tree-of-thought : Exploring INtermediate steps

In [ ]:
#explores different solutions to the problem at hand. Then votes for the best solutions and continues to the next step.
# zero shot TOT
zeroshot_tot_prompt = [
    {
        "role": "user",
        "content": "Imagine three different experts are answering this question.\
        All experts will write doen 1 step of their thinking, then share it with the group. then \
        all experts will go on to the next step, etc. If any expert realizes they're wrong at any point \
        then they leave. The question is 'The cafetaria had 23. If they used 20 to make lunch and bought 6 more, how\
        many apples do they have ?' Make sure to discuss the results."
    }
]
outputs = pipe(zeroshot_tot_prompt, use_cache=False)
print(outputs[0]["generated_text"])

 Expert 1:
Step 1: The question seems to be about the cafeteria's inventory of apples. However, the initial information provided is about the cafeteria having 23 items, but it doesn't specify what these items are.

Step 2: Assuming that the 23 items are apples, the cafeteria used 20 apples to make lunch. This leaves them with 3 apples (23 - 20 = 3).

Step 3: The cafeteria then bought 6 more apples. Adding these to the remaining 3 apples, they now have 9 apples (3 + 6 = 9).

Expert 2:
Step 1: The question is about the cafeteria's inventory of apples. The initial information provided is about the cafeteria having 23 items, but it doesn't specify what these items are.

Step 2: Assuming that the 23 items are apples, the cafeteria used 20 apples to make lunch. This leaves them with 3 apples (2


# Output Verification
## Providing Examples

In [ ]:
# zero shot learning - providing no examples
zeroshot_prompt = [
{
    "role": "user",
    "content": " Create a character profilie for an RPG game in JSON format."
}
]
outputs = pipe(zeroshot_prompt, use_cache=False)
print(outputs[0]["generated_text"])

 ```json
{
  "name": "Eldrin the Wise",
  "class": "Wizard",
  "race": "Elf",
  "level": 10,
  "attributes": {
    "strength": 10,
    "dexterity": 18,
    "constitution": 12,
    "intelligence": 20,
    "wisdom": 16,
    "charisma": 14
  },
  "skills": {
    "spellcasting": 18,
    "alchemy": 14,
    "history": 16,
    "animal_handling": 12
  },
  "equipment": {
    "weapon": "Staff of the Ancients",
    "armor": "Leather Robe",
    "accessories": ["Crystal of Power", "Ring of Protection"]
  },
  "spells": [
    {
      "name": "Fireball",
      "level": 3,



In [ ]:
# one shot learning : providing an example of the output structure
one_shot_template = """Create a short character profile for an RPG game.Make sure to only use this format:
{
  "description": "A brief description of the character",
  "name": "The character's name",
  "armor": "The character's armor class",
  "strength": "The character's strength score",
  "intelligence": "The character's intelligence score",
  "wisdom": "The character's wisdom score",
  "constitution": "The character's constitution score
  "
}"""
one_shot_prompt = [
    {
        "role": "user",
        "content": one_shot_template
    }
]
outputs = pipe(one_shot_prompt, use_cache=False)
print(outputs[0]["generated_text"])


 {
  "description": "A cunning rogue with a knack for stealth and deception, always looking for the next big score.",
  "name": "Lyra Shadowhand",
  "armor": "10 (light armor)",
  "strength": "12",
  "intelligence": "18",
  "wisdom": "14",
  "constitution": "13"
}


## Grammar: Constrained Sampling

In [23]:
# restart before running below code as a new model is being loaded - to clear the VRAM.
import gc
import torch
#del model, tokenizer, pipe

# flush memeory.
gc.collect()
torch.cuda.empty_cache()

In [24]:
!pip install llama-cpp-python
from llama_cpp.llama import Llama

# load phi-3
llm = Llama.from_pretrained(
    repo_id = "microsoft/Phi-3-mini-4k-instruct-gguf",
    filename = "*fp16.gguf",
    n_gpu_layers = -1,
    n_ctx = 2048,
    verbose = False
)

llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


In [25]:
# generate output
output = llm.create_chat_completion(
    messages = [
        {
            "role": "user",
            "content": "Create a warrior for an RPG in JSON format."
        }
    ],
    response_format = {"type": "json_object"},
    temperature = 0
)['choices'][0]['message']["content"]


In [26]:
output

'{\n  "warrior": {\n    "name": "Eldric Stormbringer",\n    "class": "Warrior",\n    "level": 5,\n    "attributes": {\n      "strength": 18,\n      "dexterity": 10,\n      "constitution": 16,\n      "intelligence": 8,\n      "wisdom": 10,\n      "charisma": 12\n    },\n    "skills": [\n      {\n        "name": "Martial Arts",\n        "proficiency": 20,\n        "description": "Expert in hand-to-hand combat and weapon handling."\n      },\n      {\n        "name": "Shield Block",\n        "proficiency": 18,\n        "description": "Highly skilled at deflecting attacks with a shield."\n      },\n      {\n        "name": "Heavy Armor",\n        "proficiency": 16,\n        "description": "Expertly equipped with heavy armor for protection."\n      },\n      {\n        "name": "Survival",\n        "proficiency": 14,\n        "description": "Adept at finding food, water, and shelter in the wilderness."\n      }\n    ],\n    "equipment": [\n      {\n        "name": "Iron Sword",\n        "typ

In [27]:
# check whether output is actually json
import json

json_output = json.dumps(json.loads(output), indent= 4)
print(json_output)

{
    "warrior": {
        "name": "Eldric Stormbringer",
        "class": "Warrior",
        "level": 5,
        "attributes": {
            "strength": 18,
            "dexterity": 10,
            "constitution": 16,
            "intelligence": 8,
            "wisdom": 10,
            "charisma": 12
        },
        "skills": [
            {
                "name": "Martial Arts",
                "proficiency": 20,
                "description": "Expert in hand-to-hand combat and weapon handling."
            },
            {
                "name": "Shield Block",
                "proficiency": 18,
                "description": "Highly skilled at deflecting attacks with a shield."
            },
            {
                "name": "Heavy Armor",
                "proficiency": 16,
                "description": "Expertly equipped with heavy armor for protection."
            },
            {
                "name": "Survival",
                "proficiency": 14,
                "